# ２値分類モデル作成

辞書データから、アルファベットなしで構成されている姓名とそのカナのみを抜き出してデータセットを作成する。
２値分類のため、辞書はすべて正例であることから、負例を同数作成する。

In [1]:
import os
ver='2.0o'
DATA=f'dataset3/dataset_j.{ver}'
os.environ['DATA']=DATA

In [2]:
# 乱数固定
import json
import torch
import json
import random
import numpy as np
import torch

def set_seed(seed=42):
    # Python
    random.seed(seed)

    # NumPy
    np.random.seed(seed)

    # PyTorch（CPU と GPU）
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # 再現性のために CuDNN を固定
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# prepare train data

In [3]:
!mkdir -p model


In [4]:
def normalize_to_fullwidth_upper(s: str) -> str:
    result = []
    for c in s:
        # 半角英字 → 大文字 → 全角
        if 'a' <= c <= 'z' or 'A' <= c <= 'Z':
            result.append(chr(ord(c.upper()) + 0xFEE0))
        # 半角数字 → 全角数字
        elif '0' <= c <= '9':
            result.append(chr(ord(c) + 0xFEE0))
        # 全角小文字英字 → 全角大文字
        elif 'ａ' <= c <= 'ｚ':
            result.append(chr(ord(c) - 0x20))
        else:
            result.append(c)
    return ''.join(result)

In [5]:
print(normalize_to_fullwidth_upper('abc1ＢＣb'))

ＡＢＣ１ＢＣＢ


In [6]:
REPLACE_MAP = {
    "ウ゛": "ヴ",
    "カ゛": "ガ",
    "キ゛": "ギ",
    "ク゛": "グ",
    "ケ゛": "ゲ",
    "コ゛": "ゴ",
    "サ゛": "ザ",
    "シ゛": "ジ",
    "ス゛": "ズ",
    "セ゛": "ゼ",
    "ソ゛": "ゾ",
    "タ゛": "ダ",
    "チ゛": "ヂ",
    "ツ゛": "ヅ",
    "テ゛": "デ",
    "ト゛": "ド",
    "ハ゛": "バ",
    "ヒ゛": "ビ",
    "フ゛": "ブ",
    "ヘ゛": "ベ",
    "ホ゛": "ボ",
}

def normalize_voiced_katakana(s: str) -> str:
    for k, v in REPLACE_MAP.items():
        s = s.replace(k, v)
    return s


LONG_SOUND_MAP = {
    '−': 'ー',
    '-': 'ー',
    '－': 'ー',
    '—': 'ー',
    '‐': 'ー',
}

def normalize_long_sound(s: str) -> str:
    return normalize_voiced_katakana(''.join(LONG_SOUND_MAP.get(c, c) for c in s))

## アルファベットを含む名前除去

In [7]:
import json

with open(f'../dict/data/data.{ver}/reliable_{ver}.json','r',encoding='utf-8') as f:
    rdic=json.load(f)

# 学習なので多少あやふやなものも入れておく
with open(f'../dict/data/data.{ver}/unreliable_{ver}.json','r',encoding='utf-8') as f:
    udic=json.load(f)

with open(f'../dict/data/data.{ver}/tankanji_{ver}.json','r',encoding='utf-8') as f:
    tdic=json.load(f)

In [8]:
import gzip
import re

pattern = re.compile(r'^[Ａ-Ｚａ-ｚ]+$')
katakana_pattern = re.compile(r'^[ァ-ヶー\s]+$')
def select_japan(jdata):
    data=[]
    for kanji,vv in jdata.items():
        for kana in vv.keys():
            kanji=normalize_to_fullwidth_upper(kanji)
            kana=normalize_long_sound(kana)

            if pattern.fullmatch(kanji):  # all alphabet
                continue
            if not katakana_pattern.fullmatch(kana):
                continue
            data.append([kanji,kana])
    return data

In [9]:
rtmp = select_japan(rdic)
utmp = select_japan(udic)

ktmp=['_'.join(l) for l in rtmp+utmp]
ktmp = list(set(ktmp))
import random
random.shuffle(ktmp)

pairs=[l.split('_') for l in ktmp]

In [10]:
siz=len(pairs)
train_idx=siz//10*8
valid_idx=train_idx  + siz//10


# tankanji
ttmp=['_'.join(l) for l in select_japan(tdic)]
ttmp = list(set(ttmp))
tpairs=[l.split('_') for l in ttmp]

train = pairs[:train_idx]+tpairs  # 訓練には単漢字を入れる
valid = pairs[train_idx:valid_idx]
test = pairs[valid_idx:]

print(len(train))
print(len(valid))
print(len(test))

1512520
174325
174334


In [11]:
tmp=[f'{a},{b}' for a,b in valid]
print(len(tmp))
print(len(list(set(tmp))))

174325
174325


In [12]:
tmp=[f'{a},{b}' for a,b in test]
print(len(tmp))
print(len(list(set(tmp))))

174334
174334


In [13]:
# unreliable
import json

with open(f'../dict/data/data.{ver}/unreliable_{ver}.json','r',encoding='utf-8') as f:
    jdata=json.load(f)

udata=[]
for k,vv in jdata.items():
    k = normalize_to_fullwidth_upper(k)


    if pattern.fullmatch(k):
        continue
    for kana in vv.keys():
        kana=normalize_long_sound(kana)
        if not katakana_pattern.fullmatch(kana):
            continue
        udata.append([k,kana])
print(len(udata))

55132


# negative set
漢字姓名とカタカナのペアをランダムに並び変えて作成する。
そのペアが信頼のあるペアにないことを確認の上、負例として付け加える

In [14]:
# 91_dataset_j.ipynb の実装内容
# ... (前段の normalization や loading は 71_dataset_j.ipynb と同様のため省略) ...

import random
import copy

class NegativeSampler:
    """
    負例の難易度を制御するクラス
    """
    def __init__(self, katakana_pool, voiced_map):
        self.katakana_pool = katakana_pool
        self.voiced_map = voiced_map
        # 濁点・半濁点を外すための逆引きマップ
        self.reverse_voiced_map = {v: k for k, v in voiced_map.items()}

    def generate_easy(self, length):
        """Level 1: ランダムな文字の羅列 (長さは同じ) """
        return "".join(random.choice(self.katakana_pool) for _ in range(length))

    def generate_medium(self, length):
        """Level 2: 長さが異なる (±1) """
        new_len = max(1, length + random.choice([-1, 1]))
        return self.generate_easy(new_len)

    def generate_hard(self, kana):
        """Level 3: 1文字だけ置換 (長さは同じ) """
        if len(kana) < 2:
            return self.generate_medium(len(kana))
        kana_list = list(kana)
        idx = random.randint(0, len(kana_list) - 1)
        kana_list[idx] = random.choice(self.katakana_pool)
        return "".join(kana_list)

    def generate_very_hard(self, kana):
        """Level 4: 濁点・拗音の微修正 (長さは同じ) """
        kana_list = list(kana)
        idx = random.randint(0, len(kana_list) - 1)
        char = kana_list[idx]

        # 濁点/半濁点の操作
        if char in self.voiced_map:
            kana_list[idx] = self.voiced_map[char]
        elif char in self.reverse_voiced_map:
            kana_list[idx] = self.reverse_voiced_map[char]
        else:
            # 拗音/小書き文字の操作 (例: ャ -> ヤ)
            # 簡易的に、関連する文字のリストを持っておくとより強力
            small_to_large = {'ャ': 'ヤ', 'ュ': 'ユ', 'ョ': 'ヨ', 'ッ': 'ツ'}
            large_to_small = {'ヤ': 'ャ', 'ユ': 'ュ', 'ヨ': 'ョ', 'ツ': 'ッ'}

            if char in small_to_large:
                kana_list[idx] = small_to_large[char]
            elif char in large_to_small:
                kana_list[idx] = large_to_small[char]
            else:
                kana_list[idx] = random.choice(self.katakana_pool)

        return "".join(kana_list)

# --- メインのデータ生成ロジック ---

# 定数定義
# NegativeSampler の voiced_map は「清音カナ -> 濁音/半濁音カナ」の対応が必要なため、
# 1文字単位のマップを用意する
VOICED_MAP = {
    "ウ": "ヴ", "カ": "ガ", "キ": "ギ", "ク": "グ", "ケ": "ゲ",
    "コ": "ゴ", "サ": "ザ", "シ": "ジ", "ス": "ズ", "セ": "ゼ",
    "ソ": "ゾ", "タ": "ダ", "チ": "ヂ", "ツ": "ヅ", "テ": "デ",
    "ト": "ド", "ハ": "バ", "ヒ": "ビ", "フ": "ブ", "ヘ": "ベ",
    "ホ": "ボ",
}

# カタカナプール（全文字）
all_kana = "アイウエオカキクケコサシスセソタチツテトナニヌネノハヒフヘホマミムメモヤユヨラリルレロワヲァィゥェォッャュョガギグゲゴザジズゼゾダヂヅデドバビブベボパピプペポ"
sampler = NegativeSampler(all_kana, VOICED_MAP)

# 既存の正解ペア（train, valid, test, udata）を取得
# (71_dataset_j.ipynb から取得した train, valid, test を使用)
okdic = {}
for ji, na in (train + udata + valid + test):
    okdic[f"{ji}_{na}"] = 1

# 正例データセット（値=1）
train_dataset = {f"{k}_{v}": 1 for k, v in train}
valid_dataset = {f"{k}_{v}": 1 for k, v in valid}
test_dataset = {f"{k}_{v}": 1 for k, v in test}

# 負例の生成設定
# 難易度別の生成比率 (Easy, Medium, Hard, VeryHard)
RATIO = [0.25, 0.25, 0.25, 0.25]


# 各データセットに対して負例を生成して追加
def add_negatives(target_dict, existing_ok_dic, ratio_list):
    # ratio_list: [easy, medium, hard, very_hard]
    # 各正例に対して、指定された比率で負例を生成する
    # ただし、既存の okdic（本物のペア）に含まれるものは負例にしない
    new_negatives = {}

    for pair_str in list(target_dict.keys()):
        ji, na = pair_str.split('_')

        for mode_idx, weight in enumerate(ratio_list):
            # 各モードについて、確率的に負例を生成
            num_samples = int(weight * 5)  # 1正例あたり、各モードで生成する最大数
            for _ in range(num_samples):
                if random.random() > 0.5:  # 確率的に生成
                    continue

                if mode_idx == 0:    # Easy
                    new_v = sampler.generate_easy(len(na))
                elif mode_idx == 1:  # Medium
                    new_v = sampler.generate_medium(len(na))
                elif mode_idx == 2:  # Hard
                    new_v = sampler.generate_hard(na)
                else:                # Very Hard
                    new_v = sampler.generate_very_hard(na)

                # 漢字は同じまま、読みだけを誤ったペアを負例にする
                new_key = f"{ji}_{new_v}"

                # 元の読みと同じ、または本物のペア・既存の正例と重複する場合は除外
                if new_v == na:
                    continue
                if new_key in existing_ok_dic:
                    continue
                if new_key in target_dict:
                    continue

                new_negatives[new_key] = 0

    return new_negatives


# 各データセットに負例を追加
train_dataset.update(add_negatives(train_dataset, okdic, RATIO))
valid_dataset.update(add_negatives(valid_dataset, okdic, RATIO))
test_dataset.update(add_negatives(test_dataset, okdic, RATIO))

print(len(train_dataset), sum(train_dataset.values()))
print(len(valid_dataset), sum(valid_dataset.values()))
print(len(test_dataset), sum(test_dataset.values()))


4492858 1504959
521481 174325
521081 174334


In [15]:
# insample check
newvalid_dataset={}
for k in valid_dataset.keys():
    if k in train_dataset:
        continue
    newvalid_dataset[k]=valid_dataset[k]

newtest_dataset={}
for k in test_dataset.keys():
    if k in train_dataset:
        continue
    newtest_dataset[k]=test_dataset[k]

In [16]:
!mkdir -p $DATA

In [17]:
## save
import gzip

def save_file(data,fname):
    with gzip.open(f'{DATA}/{fname}.csv.gz','wt',encoding='utf-8') as f:
        f.write('kanji_kana,flag\n')
        for key,flg in data.items():
            f.write(f"{key},{flg}\n")

save_file(train_dataset,'train_j')
save_file(newvalid_dataset,'valid_j')
save_file(newtest_dataset,'test_j')

In [18]:
!date

2026年  8月  5日 水曜日 13:17:49 JST
